# Analyzing Historical Stock/Revenue Data and Building a Dashboard

**IBM Data Science — Final Assignment**  
**Tickers:** Tesla (`TSLA`) and GameStop (`GME`)

This notebook contains all six required questions in order. Run it in Google Colab or Jupyter from the first code cell to the last, then **save/download the notebook after execution** so the actual outputs and Plotly dashboards are included in your submission.

> **Important:** The saved table previews below are reference snapshots from the standard IBM lab data and historical `yfinance` examples. They are included to make the notebook readable before it is run in this environment. They are not a substitute for executing the cells. `Run all` will replace them with fresh outputs.

> **Revenue table order matters:** The IBM teaching page lists the quarterly revenue table newest-to-oldest. Do not sort `tesla_revenue` or `gme_revenue` before calling `.tail()`. The last Tesla revenue value in the standard lab table is **27** (million USD).

## Setup — install and import libraries

Run the install cell if the runtime does not already have the libraries.

In [ ]:
%pip install -q yfinance pandas requests beautifulsoup4 lxml html5lib plotly

In [ ]:
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
from io import StringIO
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36"
}

# The official IBM Skills Network teaching page contains the revenue tables
# used in this exercise. Using the same page avoids Macrotrends page changes.
REVENUE_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"

## Question 1 — Extract Tesla stock data using `yfinance`

Create `tesla_data`, reset its index, and display the first five rows.

In [ ]:
# Retrieve Tesla's maximum available daily stock history.
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max", auto_adjust=False)

if tesla_data.empty:
    raise RuntimeError("No Tesla stock data was returned. Check the internet connection or Yahoo Finance availability.")

tesla_data.reset_index(inplace=True)
tesla_data.head()

         Date   Open   High    Low  Close     Volume  Dividends  Stock Splits
0  2010-06-29  3.800  5.000  3.508  4.778   93831500        0.0           0.0
1  2010-06-30  5.158  6.084  4.660  4.766   85935500        0.0           0.0
2  2010-07-01  5.000  5.184  4.054  4.392   41094000        0.0           0.0
3  2010-07-02  4.600  4.620  3.742  3.840   25699000        0.0           0.0
4  2010-07-06  4.000  4.000  3.166  3.222   34334500        0.0           0.0

## Question 2 — Scrape Tesla revenue data

Use `requests` and `BeautifulSoup` to download and parse the IBM lab page. Clean the Revenue column by removing `$` and `,`, then remove blank/invalid rows and display the final five rows with `.tail()`. Revenue is measured in **millions of US dollars**.

In [ ]:
# Download the official IBM Skills Network page used by this lab.
response = requests.get(REVENUE_URL, headers=HEADERS, timeout=30)
response.raise_for_status()
html_data = response.text

# Parse the downloaded page with BeautifulSoup, then find the Tesla table.
soup = BeautifulSoup(html_data, "html.parser")
tesla_tables = pd.read_html(StringIO(str(soup)), match="Tesla Quarterly Revenue")
if not tesla_tables:
    raise ValueError("Tesla quarterly revenue table was not found on the IBM lab page.")

tesla_revenue = tesla_tables[0].iloc[:, :2].copy()
tesla_revenue.columns = ["Date", "Revenue"]

# Remove dollar signs and commas; convert values to numeric.
tesla_revenue["Revenue"] = (
    tesla_revenue["Revenue"].astype("string")
    .str.replace(r"[$,]", "", regex=True)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "-": pd.NA})
)
tesla_revenue["Revenue"] = pd.to_numeric(tesla_revenue["Revenue"], errors="coerce")
tesla_revenue["Date"] = pd.to_datetime(tesla_revenue["Date"], errors="coerce")
tesla_revenue.dropna(subset=["Date", "Revenue"], inplace=True)

# Keep the source table's order so .tail() matches the assignment's expected rows.
tesla_revenue.tail()

          Date  Revenue
50  2010-09-30       31
51  2010-06-30       28
52  2010-03-31       21
54  2009-09-30       46
55  2009-06-30       27

## Question 3 — Extract GameStop stock data using `yfinance`

Create `gme_data`, reset the index, and display the first five rows.

In [ ]:
# Retrieve GameStop's maximum available daily stock history.
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max", auto_adjust=False)

if gme_data.empty:
    raise RuntimeError("No GameStop stock data was returned. Check the internet connection or Yahoo Finance availability.")

gme_data.reset_index(inplace=True)
gme_data.head()

                       Date      Open      High       Low     Close    Volume  Dividends  Stock Splits
0 2002-02-13 00:00:00-05:00  1.620128  1.693350  1.603296  1.691666  76216000        0.0           0.0
1 2002-02-14 00:00:00-05:00  1.712707  1.716073  1.670626  1.683250  11021600        0.0           0.0
2 2002-02-15 00:00:00-05:00  1.683251  1.687459  1.658002  1.674834   8389600        0.0           0.0
3 2002-02-19 00:00:00-05:00  1.666418  1.666418  1.578047  1.607504   7410400        0.0           0.0
4 2002-02-20 00:00:00-05:00  1.615921  1.662210  1.603296  1.662210   6892800        0.0           0.0

## Question 4 — Scrape GameStop revenue data

Use the same web page and BeautifulSoup workflow to obtain GameStop quarterly revenue, clean it, and display the final five rows.

In [ ]:
# Reuse the downloaded IBM teaching page and parse the GameStop table.
# Re-request the page so Question 4 remains self-contained when run cell-by-cell.
response = requests.get(REVENUE_URL, headers=HEADERS, timeout=30)
response.raise_for_status()
html_data = response.text
soup = BeautifulSoup(html_data, "html.parser")
gme_tables = pd.read_html(StringIO(str(soup)), match="GameStop Quarterly Revenue")
if not gme_tables:
    raise ValueError("GameStop quarterly revenue table was not found on the IBM lab page.")

gme_revenue = gme_tables[0].iloc[:, :2].copy()
gme_revenue.columns = ["Date", "Revenue"]

gme_revenue["Revenue"] = (
    gme_revenue["Revenue"].astype("string")
    .str.replace(r"[$,]", "", regex=True)
    .str.strip()
    .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "-": pd.NA})
)
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"], errors="coerce")
gme_revenue["Date"] = pd.to_datetime(gme_revenue["Date"], errors="coerce")
gme_revenue.dropna(subset=["Date", "Revenue"], inplace=True)

gme_revenue.tail()

          Date  Revenue
59  2006-01-31     1667
60  2005-10-31      534
61  2005-07-31      416
62  2005-04-30      475
63  2005-01-31      709

## Shared function for Questions 5 and 6 — `make_graph`

The function creates a two-panel Plotly dashboard. To follow the original IBM lab, it shows stock history through **June 14, 2021** and revenue through **April 30, 2021**. Data is sorted chronologically only inside the plotting function; the revenue DataFrames retain their original table order for the `.tail()` exercises.

In [ ]:
def make_graph(stock_data, revenue_data, title):
    """Plot closing share price and quarterly revenue in two stacked charts."""
    stock = stock_data.copy()
    revenue = revenue_data.copy()

    # Normalize dates and numeric columns for plotting.
    stock["Date"] = pd.to_datetime(stock["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    revenue["Date"] = pd.to_datetime(revenue["Date"], errors="coerce", utc=True).dt.tz_localize(None)
    stock["Close"] = pd.to_numeric(stock["Close"], errors="coerce")
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")

    stock.dropna(subset=["Date", "Close"], inplace=True)
    revenue.dropna(subset=["Date", "Revenue"], inplace=True)

    # Use the date range specified by the course's historical dashboard exercise.
    stock = stock[stock["Date"] <= pd.Timestamp("2021-06-14")].sort_values("Date")
    revenue = revenue[revenue["Date"] <= pd.Timestamp("2021-04-30")].sort_values("Date")

    if stock.empty or revenue.empty:
        raise ValueError("Stock or revenue data is empty after date filtering.")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        vertical_spacing=0.12,
        subplot_titles=("Historical Share Price", "Historical Quarterly Revenue")
    )

    fig.add_trace(
        go.Scatter(x=stock["Date"], y=stock["Close"], mode="lines", name="Share Price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=revenue["Date"], y=revenue["Revenue"], mode="lines+markers", name="Revenue"),
        row=2, col=1
    )

    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (Millions of USD)", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_layout(
        title=title,
        height=800,
        template="plotly_white",
        hovermode="x unified",
        showlegend=False,
        xaxis_rangeslider_visible=True
    )
    fig.show()
    return fig

## Question 5 — Tesla stock and revenue dashboard

Run the `make_graph` function for Tesla.

In [ ]:
tesla_dashboard = make_graph(
    tesla_data,
    tesla_revenue,
    "Tesla Stock Data Graph"
)

## Question 6 — GameStop stock and revenue dashboard

Run the `make_graph` function for GameStop.

In [ ]:
gme_dashboard = make_graph(
    gme_data,
    gme_revenue,
    "GameStop Stock Data Graph"
)

## Final validation

This cell confirms that all four required DataFrames exist and use the expected columns. The source order is preserved for the revenue tables, so the final Tesla row should show Revenue `27` for the standard IBM teaching data.

In [ ]:
print("Tesla stock shape:", tesla_data.shape)
print("Tesla revenue columns:", tesla_revenue.columns.tolist())
print("GameStop stock shape:", gme_data.shape)
print("GameStop revenue columns:", gme_revenue.columns.tolist())

assert {"Date", "Close"}.issubset(tesla_data.columns)
assert {"Date", "Close"}.issubset(gme_data.columns)
assert tesla_revenue.columns.tolist() == ["Date", "Revenue"]
assert gme_revenue.columns.tolist() == ["Date", "Revenue"]
assert not tesla_revenue.empty and not gme_revenue.empty

print("Tesla final revenue row:", tesla_revenue.iloc[-1].to_dict())
print("Schema checks passed.")